# MCP Tool Timeouts — Async HandleId Pattern

Based on:
- [Resilient AI Agents With MCP](https://octopus.com/blog/mcp-timeout-retry) — Octopus, May 2025
- [Call remote MCP server tool timed out, error 424](https://community.openai.com/t/call-remote-mcp-server-tool-timed-out-resulting-in-error-424/1364167) — OpenAI Community

## The Problem

MCP (Model Context Protocol) tools call external APIs. When those APIs are slow or unresponsive, the agent has no fallback — it either waits indefinitely or receives a **424 Failed Dependency** error that terminates the workflow.

The agent has no way to distinguish between "slow but working" and "hung forever". It just waits.

## The Solution: Async HandleId Pattern

Instead of blocking until the operation completes, the tool:
1. Starts the job in the background immediately
2. Returns a handle ID — e.g., `"JOB_STARTED: abc12345"` — in under 2 seconds
3. The agent calls `check_job_status(handle_id)` to poll for the result

The MCP request completes instantly. No timeouts. No 424 errors.

![Synchronous MCP Tool vs Async Pattern](../images/Synchronous-MCP-Tool.jpg)

## The MCP Tools

Six tools in `mcp_server.py` simulate real timeout scenarios from research:

| Tool | Behavior | What it represents |
|------|----------|--------------------|
| `fast_api(query)` | Responds in 1s | Normal external API — good UX baseline |
| `slow_api(query)` | Responds in 15s | Slow database query — agent waits, poor UX |
| `unresponsive_api(query)` | Never responds (300s) | API accepts request but hangs forever |
| `failing_api(query)` | Raises 424 after 5s | Failed Dependency — workflow terminates |
| `start_long_job(task)` | Returns handle immediately | **Solution**: async start — responds in < 2s |
| `check_job_status(job_id)` | Returns PROCESSING or COMPLETED | **Solution**: async poll |

![MCP Tool Response Patterns timeline](../images/MCP-Tool-Response-Patterns.png)

## What We Test

| Test | Tool | Expected time | Result |
|------|------|---------------|--------|
| 1 — Fast API | `fast_api` | ~2s | ✅ Good UX baseline |
| 2 — Slow API | `slow_api` | ~17s | ❌ Agent blocked, waits full duration |
| 3 — Failing API | `failing_api` | ~7s | ❌ 424 error, workflow fails |
| 4 — Async Pattern | `start_long_job` + `check_job_status` | ~3s | ✅ Immediate response |

## 📦 Setup

In [ ]:
import os
os.environ.setdefault('OTEL_SDK_DISABLED', 'true')
import logging, warnings  # silence OpenTelemetry 'Failed to detach context' noise
logging.getLogger('opentelemetry').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore', message='Failed to detach context')
import time
from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("⚠️ OPENAI_API_KEY not set")

# ─────────────────────────────────────────────────────────────────────────────
# How to switch the model provider (token counting works the same on all of them).
# Replace `OpenAIModel(model_id="gpt-4o-mini")` inside create_mcp_agent() with:
#
# Amazon Bedrock — uses boto3, NO extra package needed.
#   Requires AWS credentials and model access enabled in the Amazon Bedrock console.
#       from strands.models import BedrockModel
#       BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0", region_name="us-east-1")
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/
#
# Anthropic (direct API) — requires:  pip install 'strands-agents[anthropic]'
#   The API key goes inside client_args (get one at https://console.anthropic.com/).
#       from strands.models.anthropic import AnthropicModel
#       AnthropicModel(client_args={"api_key": os.getenv("ANTHROPIC_API_KEY")},
#                      model_id="claude-sonnet-4-6", max_tokens=1028)
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/anthropic/
# ─────────────────────────────────────────────────────────────────────────────
def create_mcp_agent():
    """Create agent with MCP tools."""
    mcp_client = MCPClient(
        lambda: stdio_client(
            StdioServerParameters(
                command="python",
                args=["mcp_server.py"]
            )
        )
    )
    
    return Agent(
        model=OpenAIModel(model_id="gpt-4o-mini"),
        tools=[mcp_client]
    )

print("✅ Setup complete!")

---

## 🔬 Scenario 1: Fast API (Baseline)

**Research:** Fast APIs provide good UX

**Expected:** Quick response (~3-5s)

In [ ]:
agent = create_mcp_agent()
query = "Use fast_api to process 'user data'"

start = time.time()
response = agent(query)
time_fast = time.time() - start

print(f"Response: {response}")
print(f"⏱️  {time_fast:.1f}s")
# 📊 Token counting — Strands native metric (same for OpenAI, Bedrock, etc.).
# result.metrics.accumulated_usage sums ALL the LLM calls made during this task:
#   inputTokens  = tokens sent (prompt + system prompt + tool definitions + history)
#   outputTokens = tokens generated by the model
#   totalTokens  = input + output
# We check '.metrics' because it can be None if the provider does not report usage.
tokens_fast = response.metrics.accumulated_usage['totalTokens'] if response.metrics else 0
print(f"💰 Tokens: {tokens_fast:,} total")


---

## ⚠️ Scenario 2: Slow API (Problem)

**Research Finding:** "Agent waits indefinitely - No timeout configured"

**Expected:** Agent waits 15+ seconds - poor UX

In [ ]:
agent = create_mcp_agent()
query = "Use slow_api to query database for 'customer records'"

print("⏳ Waiting ~15s for slow API...\n")

start = time.time()
response = agent(query)
time_slow = time.time() - start

print(f"Response: {response}")
print(f"⏱️  {time_slow:.1f}s — agent waited full duration")
# Tokens for this call (Strands native metric, see explanation above).
tokens_slow = response.metrics.accumulated_usage['totalTokens'] if response.metrics else 0
print(f"💰 Tokens: {tokens_slow:,} total")


---

## ❌ Scenario 3: Failing API (424 Error)

**Research:** "424 Failed Dependency when MCP tools timeout"

**Expected:** Error after delay

In [ ]:
agent = create_mcp_agent()
query = "Use failing_api to connect to external service"

start = time.time()
response = agent(query)
time_failing = time.time() - start

print(f"Response: {response}")
print(f"⏱️  {time_failing:.1f}s")
# Tokens for this call (Strands native metric, see explanation above).
tokens_failing = response.metrics.accumulated_usage['totalTokens'] if response.metrics else 0
print(f"💰 Tokens: {tokens_failing:,} total")


---

## 💡 Scenario 4: Async Pattern (Solution)

**Research Solution:** "Return handleId immediately, check status later"

**Expected:** Immediate response, good UX

In [ ]:
agent = create_mcp_agent()

# Step 1: start job → returns handle immediately
query1 = "Use start_long_job to process 'large dataset'"
start = time.time()
response1 = agent(query1)
elapsed1 = time.time() - start

print(f"Step 1 — handle returned in {elapsed1:.1f}s")
print(f"Response: {response1}")
# Tokens for this call (Strands native metric, see explanation above).
tokens_async1 = response1.metrics.accumulated_usage['totalTokens'] if response1.metrics else 0
print(f"💰 Tokens: {tokens_async1:,} total")


In [ ]:
# Step 2: check status
query2 = "Use check_job_status to check the job that was just started"
start = time.time()
response2 = agent(query2)
elapsed2 = time.time() - start
time_async = elapsed1 + elapsed2

print(f"Step 2 — status in {elapsed2:.1f}s  |  Total: {time_async:.1f}s")
print(f"Response: {response2}")
# Tokens for this call (Strands native metric, see explanation above).
tokens_async2 = response2.metrics.accumulated_usage['totalTokens'] if response2.metrics else 0
print(f"💰 Tokens: {tokens_async2:,} total")


In [ ]:
improvement = time_slow - time_async
pct = 100 * improvement / time_slow

print(f"{'Scenario':<30} {'Time':>8}  UX")
print("-"*50)
print(f"{'1. Fast API (baseline)':<30} {time_fast:>6.1f}s  ✅")
print(f"{'2. Slow API (problem)':<30} {time_slow:>6.1f}s  ❌  waited")
print(f"{'3. Failing API (424)':<30} {time_failing:>6.1f}s  ❌  error")
print(f"{'4. Async handleId':<30} {time_async:>6.1f}s  ✅")
print(f"\n→ Async saved {improvement:.1f}s vs slow API ({pct:.0f}% faster)")
print(f"→ First response in {elapsed1:.1f}s — no timeout risk")

tokens_async = tokens_async1 + tokens_async2
print()
print(f"{'Scenario':<30} {'Tokens':>10}")
print('-'*42)
print(f"{'1. Fast API (baseline)':<30} {tokens_fast:>10,}")
print(f"{'2. Slow API (problem)':<30} {tokens_slow:>10,}")
print(f"{'3. Failing API (424)':<30} {tokens_failing:>10,}")
print(f"{'4. Async handleId':<30} {tokens_async:>10,}")


---

## References

- [Resilient AI Agents With MCP](https://octopus.com/blog/mcp-timeout-retry) — Octopus, May 2025
- [Call remote MCP server tool timed out, error 424](https://community.openai.com/t/call-remote-mcp-server-tool-timed-out-resulting-in-error-424/1364167) — OpenAI Community
- [Handling Timeouts with Long-Running MCP Connectors](https://community.openai.com/t/handling-timeouts-with-long-running-mcp-connectors-vertex-ai-agent/1369341) — OpenAI Community, Dec 2025
- [Strands MCP Tools](https://strandsagents.com/docs/user-guide/concepts/tools/mcp-tools/) — MCP integration docs